## Imports & utilities, Installing dependencies

In [1]:
!pip install mlxtend --quiet



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import itertools
import time
from collections import defaultdict
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import os
import sys

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:

# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

DATASET_FOLDER = 'data2'

if not os.path.isdir(DATASET_FOLDER):
    print("Note: DATASET_FOLDER path does not exist. Update DATASET_FOLDER variable to the path containing your CSV files.")
else:
    print("DATASET_FOLDER OK:", DATASET_FOLDER)


DATASET_FOLDER OK: data2


## Dataset Mapping

In [4]:
DATASET_FILES = {
    "Amazon": os.path.join(DATASET_FOLDER, "amazon_transactions.csv"),
    "Shoprite": os.path.join(DATASET_FOLDER, "shoprite_transactions.csv"),
    "BestBuy": os.path.join(DATASET_FOLDER, "bestbuy_transactions.csv"),
    "Adidas": os.path.join(DATASET_FOLDER, "adidas_transactions.csv"),
    "Dicks": os.path.join(DATASET_FOLDER, "dicks_sporting_transactions.csv")
}

print("Datasets available:")
for i, name in enumerate(DATASET_FILES.keys(), start=1):
    print(f"{i}. {name} -> {DATASET_FILES[name]}")


Datasets available:
1. Amazon -> data2\amazon_transactions.csv
2. Shoprite -> data2\shoprite_transactions.csv
3. BestBuy -> data2\bestbuy_transactions.csv
4. Adidas -> data2\adidas_transactions.csv
5. Dicks -> data2\dicks_sporting_transactions.csv


In [5]:
def load_transactions_from_csv(path, tx_col='transaction_list', id_col='transaction_id'):
    """
    Load transactions CSV where each row has a tx id and a comma-separated list of items.
    Returns: list of transactions, each transaction is a list of stripped item names.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{path} not found.")
    df = pd.read_csv(path, dtype=str).fillna('')
    if tx_col not in df.columns:
        raise ValueError(f"Expected column '{tx_col}' in {path}. Found columns: {df.columns.tolist()}")
    transactions = []
    for val in df[tx_col].astype(str):
        # split by comma, strip whitespace, ignore empty
        items = [it.strip() for it in val.split(',') if it.strip()!='']
        transactions.append(items)
    return transactions

# quick test (uncomment to test one file if present)
# example_transactions = load_transactions_from_csv(DATASET_FILES['Amazon'])
# print(len(example_transactions), example_transactions[:3])


21 [['Laptop', 'Wireless Earbuds', 'Portable Charger'], ['Smartphone', 'USB-C Cable', 'Portable Charger'], ['Kindle', 'Bluetooth Speaker']]


## Brute Force Algo

In [6]:
def brute_force_frequent_itemsets(transactions, min_support, verbose=False):
    """
    Brute-force enumeration of k-itemsets.
    transactions: list of lists (items)
    min_support: fraction (0..1)
    Returns:
      freq_itemsets: dict: k -> dict{ itemset_tuple(sorted): support_count }
      support_counts_all: dict mapping frozenset(itemset)->count
    """
    n_trans = len(transactions)
    min_count = max(1, int(min_support * n_trans + 1e-9))  # require at least this many occurrences
    # get universe of items
    items = sorted({it for tx in transactions for it in tx})
    item_to_index = {it:i for i,it in enumerate(items)}
    # prepare boolean matrix-like for counting faster (list of sets)
    tx_sets = [set(tx) for tx in transactions]
    support_counts_all = {}
    freq_itemsets = {}
    k = 1
    found_any = True
    while found_any:
        found_any = False
        if k == 1:
            candidates = [(it,) for it in items]
        else:
            # generate all combinations of universe items choose k
            candidates = list(itertools.combinations(items, k))
        counts = {}
        # count support for each candidate
        for cand in candidates:
            cand_set = set(cand)
            c = 0
            for tx in tx_sets:
                # membership test
                if cand_set.issubset(tx):
                    c += 1
            if c >= min_count:
                counts[tuple(cand)] = c
                support_counts_all[frozenset(cand)] = c
        if counts:
            freq_itemsets[k] = counts
            found_any = True
            if verbose:
                print(f"Found {len(counts)} frequent {k}-itemsets (min_count={min_count})")
            k += 1
        else:
            # no frequent k-itemsets found -> terminate brute force
            if verbose:
                print(f"No frequent {k}-itemsets found. Terminating.")
            break
    return freq_itemsets, support_counts_all, n_trans


In [7]:
def generate_rules_from_freq_itemsets(freq_itemsets, support_counts_all, n_trans, min_confidence):
    """
    freq_itemsets: dict k -> { tuple(itemset): count }
    support_counts_all: dict frozenset->count
    n_trans: total transactions
    min_confidence: fraction (0..1)
    Returns list of rules as dicts: {antecedent, consequent, support, confidence, support_count}
    """
    rules = []
    # iterate all frequent itemsets of size >=2
    for k, items_dict in freq_itemsets.items():
        if k < 2:
            continue
        for itemset_tuple, count in items_dict.items():
            itemset = set(itemset_tuple)
            sup_itemset = count / n_trans
            # generate all non-empty proper subsets as antecedent
            # for size s from 1..k-1
            for s in range(1, k):
                for antecedent in itertools.combinations(itemset_tuple, s):
                    antecedent = set(antecedent)
                    consequent = itemset - antecedent
                    ante_count = support_counts_all.get(frozenset(antecedent), 0)
                    if ante_count == 0:
                        continue
                    confidence = count / ante_count
                    if confidence + 1e-12 >= min_confidence:
                        rules.append({
                            "antecedent": tuple(sorted(antecedent)),
                            "consequent": tuple(sorted(consequent)),
                            "support": sup_itemset,
                            "support_count": count,
                            "confidence": confidence
                        })
    # sort rules by confidence desc then support desc
    rules = sorted(rules, key=lambda r: (-r['confidence'], -r['support']))
    return rules


## All algortihms combined

In [8]:
def transactions_to_onehot_df(transactions):
    te = TransactionEncoder()
    te_ary = te.fit(transactions).transform(transactions)
    df = pd.DataFrame(te_ary, columns=te.columns_)
    # convert booleans to ints
    return df.astype(int)


In [9]:
def normalize_support_conf(val_str, total_transactions=None):
    """
    Accepts string input; returns float between 0..1.
    Accepts '0.2' or '20' (interpreted as percent).
    """
    v = None
    try:
        v = float(val_str)
    except:
        raise ValueError("Not a number")
    if v <= 0:
        raise ValueError("Value must be > 0.")
    if v <= 1:
        return v
    # interpret values >1 up to 100 as percent
    if 1 < v <= 100:
        return v / 100.0
    return v  # fallback

def run_for_dataset(dataset_name, tx_file, min_support, min_confidence, verbose=True, run_library_algos=True):
    print(f"\n=== Running on dataset: {dataset_name} ===")
    transactions = load_transactions_from_csv(tx_file)
    print(f"Loaded {len(transactions)} transactions.\n")

    # --- Brute-force frequent itemsets ---
    t0 = time.time()
    freq_itemsets, support_counts_all, n_trans = brute_force_frequent_itemsets(
        transactions, min_support, verbose=verbose
    )
    t1 = time.time()
    brute_time = t1 - t0
    total_freq_count = sum(len(v) for v in freq_itemsets.values())

    # Generate rules
    t0 = time.time()
    rules = generate_rules_from_freq_itemsets(freq_itemsets, support_counts_all, n_trans, min_confidence)
    t1 = time.time()
    rules_time = t1 - t0

    print(f"[✓] Brute-force completed in {brute_time:.2f}s")
    print(f"Found {total_freq_count} frequent itemsets and {len(rules)} association rules.\n")

    # --- Display Brute-force Results ---
    print("=== Brute-force Results ===")
    print("Frequent Itemsets:")
    for k in sorted(freq_itemsets.keys()):
        for itemset in freq_itemsets[k]:
            support = support_counts_all[frozenset(itemset)] / n_trans
            print(f"{{{', '.join(itemset)}}} (support={support:.2f})")

    print("\nAssociation Rules:")
    if len(rules) == 0:
        print("No rules found.")
    else:
        for r in rules:
            ant = ', '.join(r['antecedent'])
            cons = ', '.join(r['consequent'])
            print(f"{ant} → {cons} (confidence={r['confidence']:.2f})")

    # --- Library-based algorithms (Apriori & FP-Growth) ---
    lib_results = {}
    if run_library_algos:
        df_onehot = transactions_to_onehot_df(transactions)

        # --- Apriori ---
        t0 = time.time()
        apr = apriori(df_onehot, min_support=min_support, use_colnames=True)
        t1 = time.time()
        apr_time = t1 - t0
        apr_rules = association_rules(apr, metric="confidence", min_threshold=min_confidence) if not apr.empty else pd.DataFrame()
        lib_results['apriori'] = {'freq_itemsets': apr, 'rules': apr_rules, 'time': apr_time}

        # --- FP-Growth ---
        t0 = time.time()
        fpg = fpgrowth(df_onehot, min_support=min_support, use_colnames=True)
        t1 = time.time()
        fpg_time = t1 - t0
        fpg_rules = association_rules(fpg, metric="confidence", min_threshold=min_confidence) if not fpg.empty else pd.DataFrame()
        lib_results['fpgrowth'] = {'freq_itemsets': fpg, 'rules': fpg_rules, 'time': fpg_time}

        # --- Apriori output ---
        print("\n=== Apriori: Frequent Itemsets ===")
        for _, row in apr.iterrows():
            items = ', '.join(list(row['itemsets']))
            print(f"{{{items}}} (support={row['support']:.2f})")

        print("\n=== Apriori: Association Rules ===")
        if apr_rules.empty:
            print("No rules found.")
        else:
            for _, row in apr_rules.iterrows():
                ant = ', '.join(list(row['antecedents']))
                cons = ', '.join(list(row['consequents']))
                print(f"{ant} → {cons} (confidence={row['confidence']:.2f})")

        # --- FP-Growth output ---
        print("\n=== FP-Growth: Frequent Itemsets ===")
        for _, row in fpg.iterrows():
            items = ', '.join(list(row['itemsets']))
            print(f"{{{items}}} (support={row['support']:.2f})")

        print("\n=== FP-Growth: Association Rules ===")
        if fpg_rules.empty:
            print("No rules found.")
        else:
            for _, row in fpg_rules.iterrows():
                ant = ', '.join(list(row['antecedents']))
                cons = ', '.join(list(row['consequents']))
                print(f"{ant} → {cons} (confidence={row['confidence']:.2f})")

        # --- Summary ---
        print("\n=== Summary ===")
        print(f"Brute-force: {total_freq_count} itemsets, {len(rules)} rules, {brute_time:.2f}s")
        print(f"Apriori: {len(apr)} itemsets, {len(apr_rules)} rules, {apr_time:.2f}s")
        print(f"FP-Growth: {len(fpg)} itemsets, {len(fpg_rules)} rules, {fpg_time:.2f}s")

    # --- Return dictionary with results ---
    return {
        'transactions': transactions,
        'freq_itemsets': freq_itemsets,
        'support_counts_all': support_counts_all,
        'n_trans': n_trans,
        'rules': rules,
        'timings': {'brute_total': brute_time, 'rules_time': rules_time},
        'lib_results': lib_results
    }

# Interactive menu & input validation
def interactive_run():
    # show dataset menu
    names = list(DATASET_FILES.keys())
    for i, nm in enumerate(names, start=1):
        print(f"{i}. {nm}")
    # prompt for selection
    while True:
        sel = input(f"Choose dataset by number (1-{len(names)}): ").strip()
        if not sel.isdigit():
            print("Please enter a number.")
            continue
        si = int(sel)
        if 1 <= si <= len(names):
            chosen = names[si-1]
            break
        else:
            print("Invalid choice, try again.")
    # get support
    while True:
        s_in = input("Enter minimum support (fraction e.g., 0.2 or percent e.g., 20): ").strip()
        try:
            ms = normalize_support_conf(s_in)
            if not (0 < ms <= 1):
                raise ValueError()
            break
        except Exception as e:
            print("Invalid support:", e)
    # get confidence
    while True:
        c_in = input("Enter minimum confidence (fraction e.g., 0.6 or percent e.g., 60): ").strip()
        try:
            mc = normalize_support_conf(c_in)
            if not (0 < mc <= 1):
                raise ValueError()
            break
        except Exception as e:
            print("Invalid confidence:", e)
    print(f"Selected dataset: {chosen}, min_support={ms}, min_confidence={mc}")
    # run experiment
    results = run_for_dataset(chosen, DATASET_FILES[chosen], ms, mc, verbose=True, run_library_algos=True)
    return results

# To run interactively (uncomment below and run)
results = interactive_run()


1. Amazon
2. Shoprite
3. BestBuy
4. Adidas
5. Dicks


Choose dataset by number (1-5):  1
Enter minimum support (fraction e.g., 0.2 or percent e.g., 20):  0.2
Enter minimum confidence (fraction e.g., 0.6 or percent e.g., 60):  0.2


Selected dataset: Amazon, min_support=0.2, min_confidence=0.2

=== Running on dataset: Amazon ===
Loaded 21 transactions.

Found 9 frequent 1-itemsets (min_count=4)
Found 1 frequent 2-itemsets (min_count=4)
No frequent 3-itemsets found. Terminating.
[✓] Brute-force completed in 0.00s
Found 10 frequent itemsets and 2 association rules.

=== Brute-force Results ===
Frequent Itemsets:
{Bluetooth Speaker} (support=0.33)
{Fire TV Stick} (support=0.33)
{Kindle} (support=0.24)
{Laptop} (support=0.38)
{Portable Charger} (support=0.33)
{Smartphone} (support=0.33)
{Smartwatch} (support=0.19)
{USB-C Cable} (support=0.33)
{Wireless Earbuds} (support=0.19)
{Bluetooth Speaker, Fire TV Stick} (support=0.19)

Association Rules:
Bluetooth Speaker → Fire TV Stick (confidence=0.57)
Fire TV Stick → Bluetooth Speaker (confidence=0.57)

=== Apriori: Frequent Itemsets ===
{Bluetooth Speaker} (support=0.33)
{Fire TV Stick} (support=0.33)
{Kindle} (support=0.24)
{Laptop} (support=0.38)
{Portable Charger} (supp